# Barbershop — standalone epoch30 validation review
เปิดใน **Colab runtime ใหม่ เลือก GPU แล้ว Run all** ไม่ต้องต่อท้าย notebook เดิม และไม่ต้องอัปโหลด ZIP

โหลด baseline pretrained + A–D epoch30 ที่ฝึกไว้ ประเมิน validation 9 คู่เดิม แล้วสร้าง CSV, กราฟ, ภาพเต็ม, ภาพขยาย และ PDF/ZIP ใน Drive โฟลเดอร์ใหม่

ไม่ฝึกซ้ำ ไม่เลือก best ใหม่ ไม่แตะ test และไม่เปลี่ยน `best.pt` เดิม การดู epoch30 ครั้งนี้เป็นการตรวจเพิ่มเติมหลังเห็นผล ไม่ใช่การเปลี่ยนกติกาย้อนหลัง

ต้องมี Drive เดิมที่เก็บ checkpoint, latent cache และ masks ครบ โค้ดจะตรวจไฟล์ก่อนโหลดโมเดล หากย้ายโฟลเดอร์ให้แก้เฉพาะช่อง path ด้านล่าง

ค่ารายละเอียดวัดความเปลี่ยนแปลงจาก baseline ไม่ใช่คะแนนคุณภาพแท้จริง และ shape target เดิมมาจาก HairFastGAN จึงเป็น proxy ที่มีข้อจำกัด ไม่ใช่ ground truth อิสระ


In [ ]:
from google.colab import drive
from pathlib import Path
import json, hashlib, os, sys, shutil, subprocess, time, copy, gc

drive.mount('/content/drive')
FT_ROOT = Path('/content/drive/MyDrive/Barbershop_FT_30Epoch/27f7f146ef0d')
STUDY_ROOT = Path('/content/drive/MyDrive/HairFastGAN_BPP_4Trials_PretrainedStart/9be83b9c2e')
BARBER_WEIGHTS = Path('/content/drive/MyDrive/Barbershop_matched_weights')
DRIVE_CACHE_DIR = Path('/content/drive/MyDrive/HairFastGAN_cache/pretrained_models')
SOURCE_OVERRIDE = ''  # เว้นว่างเพื่ออ่านตำแหน่ง dataset cache จากการทดลองเดิม

def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024**2),b''):h.update(chunk)
    return h.hexdigest()

def require(path):
    if not path.is_file():raise FileNotFoundError(f'ไม่พบไฟล์เดิม: {path} — ตรวจ path ด้านบนหรือกู้ไฟล์เดิม; ไม่สร้าง cache ใหม่แทน')
    return path
FT_PROTOCOL=json.loads(require(FT_ROOT/'protocol.json').read_text())
FT_ID=FT_ROOT.name;FT_TRIALS=FT_PROTOCOL['trials'];VAL_KEYS=FT_PROTOCOL['validation_keys']
source_manifest=json.loads(require(STUDY_ROOT/'trial_A/manifest.json').read_text())
SOURCE=Path(SOURCE_OVERRIDE or source_manifest['source'])
assert file_sha(require(SOURCE/'manifest.json'))==FT_PROTOCOL['source_manifest_sha256'],'Dataset manifest changed'
assert file_sha(require(BARBER_WEIGHTS/'ffhq.pt'))==FT_PROTOCOL['ffhq_sha256'],'Pretrained weights differ'
BARBER_ROOT=FT_ROOT/'front_end';CROPS=BARBER_ROOT/'shared_crops'
BARBER_CASES={'validation':json.loads(require(BARBER_ROOT/'validation/worker_config.json').read_text())['cases']}
assert len(VAL_KEYS)==9 and [r['key'] for r in BARBER_CASES['validation']]==VAL_KEYS
for key in VAL_KEYS:
    require(SOURCE/'cache'/(key+'.pt'))
    for name in FT_PROTOCOL['mask_hashes'][key]:require(STUDY_ROOT/'trial_A/inputs'/key/(name+'.png'))
for t in 'ABCD':
    folder=FT_ROOT/('trial_'+t)
    m=json.loads(require(folder/'manifest.json').read_text())
    assert m['completed_epoch']==30 and m['weight_updates']==4500,f'Trial {t} incomplete'
    require(folder/'final_epoch30.pt' if (folder/'final_epoch30.pt').is_file() else folder/'last.pt')
records=json.loads(require(BARBER_ROOT/'validation/results.json').read_text())
for rec in records:
    require(Path(rec['final_float']))
    assert len(list((Path(rec['final_png']).parent/'Blend_realistic').glob('*.npz')))==1,'Missing original latent cache'
for row in BARBER_CASES['validation']:
    for role in ['face','shape','color']:require(CROPS/(row[role]+'.png'))
print('พบข้อมูล validation และ checkpoint ครบ — เริ่มติดตั้งส่วนประเมิน')


## 2. ติดตั้งและโหลดตัวประเมินเดิม
ส่วนนี้ใช้ repository commit และ native operators เดียวกับ notebook เดิม รุ่น package ใน Colab อาจเปลี่ยนได้ รายงานจะบันทึก environment ของรอบนี้

In [ ]:
%pip install -q opencv-python matplotlib scipy scikit-image scikit-learn pandas tqdm dlib face-alignment gdown addict wandb torchmetrics dill ninja transformers diffusers accelerate fastapi uvicorn python-multipart pyngrok nest-asyncio huggingface_hub
%pip install -q git+https://github.com/openai/CLIP.git
import torch
assert torch.cuda.is_available(), 'เลือก Runtime > Change runtime type > GPU ก่อนรัน'
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)


In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess, sys, json, time
from huggingface_hub import snapshot_download


LOCAL_REPO = '/content/HairFastGAN_epoch30_review'
DRIVE_CACHE_DIR = Path('/content/drive/MyDrive/HairFastGAN_cache/pretrained_models')

# ใส่ commit ที่บันทึกใน metadata.json เมื่อต้องการทำซ้ำภายหลัง
HAIRFAST_REVISION = '49e98019c9243d1991c19d45764fc3340a498de2'
if not Path(LOCAL_REPO, '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AIRI-Institute/HairFastGAN.git', LOCAL_REPO], check=True)
if HAIRFAST_REVISION:
    subprocess.run(['git', '-C', LOCAL_REPO, 'checkout', HAIRFAST_REVISION], check=True)
os.chdir(LOCAL_REPO)
local_models = Path(LOCAL_REPO, 'pretrained_models')
if not local_models.exists() or not any(local_models.iterdir()):
    if DRIVE_CACHE_DIR.exists() and any(DRIVE_CACHE_DIR.iterdir()):
        shutil.copytree(DRIVE_CACHE_DIR, local_models, dirs_exist_ok=True)
    else:
        snapshot = snapshot_download('AIRI-Institute/HairFastGAN', allow_patterns=['pretrained_models/**'])
        shutil.copytree(Path(snapshot, 'pretrained_models'), local_models, dirs_exist_ok=True)

# ตรวจ cache สำคัญ รวมถึงกรณีได้ Git-LFS pointer แทน weights
for relative in ['StyleGAN/ffhq.pt', 'encoder4editing/e4e_ffhq_encode.pt']:
    p = local_models / relative
    if not p.exists() or p.stat().st_size < 1000000:
        raise RuntimeError(f'โมเดลขาดหรือไม่สมบูรณ์: {p} โปรดตรวจ cache ก่อนดำเนินการ')
Path('input').mkdir(exist_ok=True)
Path('output').mkdir(exist_ok=True)
REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repo commit:', REPO_COMMIT)


In [ ]:
# Compatibility backend: native torch operators also execute on CUDA tensors.
# No PyTorch/CUDA version downgrade; no changes to weights or transfer parameters.
import ast, hashlib, importlib, sys
from pathlib import Path
STYLEGAN_OPS_BACKEND = 'torch_native'

def _native_stylegan_source(source, kind):
    marker = '# HAIRFAST_TORCH_NATIVE_V1'
    if marker in source:
        return source
    tree = ast.parse(source)
    extension = 'fused' if kind == 'fused_act' else 'upfirdn2d_op'
    entry = 'fused_leaky_relu' if kind == 'fused_act' else 'upfirdn2d'
    edits = []
    found_load = found_entry = False
    for node in tree.body:
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == extension for t in node.targets):
            if not isinstance(node.value, ast.Call) or not isinstance(node.value.func, ast.Name) or node.value.func.id != 'load':
                raise RuntimeError('รูปแบบ extension เปลี่ยนไป: ' + kind)
            edits.append((node.lineno-1, node.end_lineno, extension + ' = None\n'))
            found_load = True
        if isinstance(node, ast.FunctionDef) and node.name == entry:
            edits.append((node.lineno-1, node.end_lineno, FUSED_NATIVE if kind == 'fused_act' else UP_NATIVE))
            found_entry = True
    if not found_load or not found_entry:
        raise RuntimeError('ไม่พบจุด patch ที่คาดไว้ใน ' + kind)
    lines = source.splitlines(True)
    for start,end,text in sorted(edits, reverse=True):
        lines[start:end] = [text]
    updated = marker + '\n' + ''.join(lines)
    if kind == 'upfirdn2d' and not any(isinstance(n, ast.FunctionDef) and n.name == 'upfirdn2d_native' for n in tree.body):
        updated += '\n\n' + UP_NATIVE_IMPL + '\n'
    compile(updated, kind + '.py', 'exec')
    return updated

FUSED_NATIVE = 'def fused_leaky_relu(input, bias=None, negative_slope=0.2, scale=2 ** 0.5):\n    if bias is not None:\n        input = input + bias.reshape(1, -1, *([1] * (input.ndim - 2)))\n    return torch.nn.functional.leaky_relu(input, negative_slope=negative_slope) * scale\n'
UP_NATIVE = 'def upfirdn2d(input, kernel, up=1, down=1, pad=(0, 0)):\n    return upfirdn2d_native(input, kernel, up, up, down, down,\n                           pad[0], pad[1], pad[0], pad[1])\n'
UP_NATIVE_IMPL = 'def upfirdn2d_native(\n    input, kernel, up_x, up_y, down_x, down_y, pad_x0, pad_x1, pad_y0, pad_y1\n):\n    _, channel, in_h, in_w = input.shape\n    input = input.reshape(-1, in_h, in_w, 1)\n\n    _, in_h, in_w, minor = input.shape\n    kernel_h, kernel_w = kernel.shape\n\n    out = input.view(-1, in_h, 1, in_w, 1, minor)\n    out = F.pad(out, [0, 0, 0, up_x - 1, 0, 0, 0, up_y - 1])\n    out = out.view(-1, in_h * up_y, in_w * up_x, minor)\n\n    out = F.pad(\n        out, [0, 0, max(pad_x0, 0), max(pad_x1, 0), max(pad_y0, 0), max(pad_y1, 0)]\n    )\n    out = out[\n        :,\n        max(-pad_y0, 0) : out.shape[1] - max(-pad_y1, 0),\n        max(-pad_x0, 0) : out.shape[2] - max(-pad_x1, 0),\n        :,\n    ]\n\n    out = out.permute(0, 3, 1, 2)\n    out = out.reshape(\n        [-1, 1, in_h * up_y + pad_y0 + pad_y1, in_w * up_x + pad_x0 + pad_x1]\n    )\n    w = torch.flip(kernel, [0, 1]).view(1, 1, kernel_h, kernel_w)\n    out = F.conv2d(out, w)\n    out = out.reshape(\n        -1,\n        minor,\n        in_h * up_y + pad_y0 + pad_y1 - kernel_h + 1,\n        in_w * up_x + pad_x0 + pad_x1 - kernel_w + 1,\n    )\n    out = out.permute(0, 2, 3, 1)\n    out = out[:, ::down_y, ::down_x, :]\n\n    out_h = (in_h * up_y + pad_y0 + pad_y1 - kernel_h) // down_y + 1\n    out_w = (in_w * up_x + pad_x0 + pad_x1 - kernel_w) // down_x + 1\n\n    return out.view(-1, channel, out_h, out_w)'

def install_native_stylegan_ops(repo):
    root = Path(repo).resolve()
    pairs = sorted((root / 'models').rglob('op/fused_act.py'))
    if not pairs:
        raise RuntimeError('ไม่พบ StyleGAN operator ใน ' + str(root))
    planned = []
    for fused_path in pairs:
        for path,kind in [(fused_path,'fused_act'), (fused_path.with_name('upfirdn2d.py'),'upfirdn2d')]:
            if not path.is_file():
                raise RuntimeError('ขาด operator: ' + str(path))
            module_name = '.'.join(path.relative_to(root).with_suffix('').parts)
            if module_name in sys.modules:
                raise RuntimeError('StyleGAN ถูก import แล้ว: Restart runtime แล้วรันตามลำดับใหม่')
            old = path.read_text()
            planned.append((path, old, _native_stylegan_source(old,kind)))
    # Validate every candidate before writing; preserve each original once.
    records = []
    for path,old,new in planned:
        if old != new:
            backup = path.with_suffix('.py.before_native')
            if not backup.exists(): backup.write_text(old)
            path.write_text(new)
        records.append({'path':str(path.relative_to(root)),
                        'sha256':hashlib.sha256(new.encode()).hexdigest()})
    importlib.invalidate_caches()
    return records

STYLEGAN_NATIVE_PATCHES = install_native_stylegan_ops(LOCAL_REPO)
print('StyleGAN backend:', STYLEGAN_OPS_BACKEND, '| patched files:', len(STYLEGAN_NATIVE_PATCHES))



In [ ]:
from PIL import Image
if not hasattr(Image,"ANTIALIAS"): Image.ANTIALIAS=Image.Resampling.LANCZOS
import sys, torch
sys.path.insert(0, LOCAL_REPO)
from hair_swap import HairFast, get_parser
hair_fast = HairFast(get_parser().parse_args([]))
print('HairFastGAN พร้อม')

import numpy as np, pandas as pd
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from models.Net import get_segmentation as segment
from IPython.display import display
b=hair_fast.blend;enc=b.blending_encoder;G=b.net.generator;pp=b.post_process;align=hair_fast.align
for module in [enc,G,pp,align]:module.eval().requires_grad_(False)


In [ ]:
runtime_baseS={}
def cpu(x):
    if torch.is_tensor(x):
        return x.detach().cpu().clone()
    if isinstance(x, dict):
        return {k: cpu(v) for k, v in x.items()}
    if isinstance(x, list):
        return [cpu(v) for v in x]
    if isinstance(x, tuple):
        return tuple((cpu(v) for v in x))
    return copy.deepcopy(x)

def gpu(x):
    if torch.is_tensor(x):
        return x.to('cuda').clone()
    if isinstance(x, dict):
        return {k: gpu(v) for k, v in x.items()}
    if isinstance(x, list):return [gpu(v) for v in x]
    if isinstance(x, tuple):return tuple(gpu(v) for v in x)
    return x

def resize(x):
    return F.interpolate(x, size=(256, 256), mode='bilinear', align_corners=False)

def saveim(x, p):
    a = x.detach().float().cpu()[0].clamp(0, 1).permute(1, 2, 0).numpy()
    Image.fromarray((a * 255).round().astype('uint8')).save(p)

def lab(x):
    x = x.float().clamp(0, 1)
    lin = torch.where(x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055).pow(2.4))
    M = x.new_tensor([[0.4124564, 0.3575761, 0.1804375], [0.2126729, 0.7151522, 0.072175], [0.0193339, 0.119192, 0.9503041]])
    xyz = torch.einsum('ij,bjhw->bihw', M, lin) / x.new_tensor([0.95047, 1.0, 1.08883])[None, :, None, None]
    f = torch.where(xyz > 0.008856, xyz.clamp_min(1e-08).pow(1 / 3), 7.787 * xyz + 16 / 116)
    return torch.stack([116 * f[:, 1] - 16, 500 * (f[:, 0] - f[:, 1]), 200 * (f[:, 1] - f[:, 2])], 1)



def detail(x):
    y = x.mean(1, keepdim=True)
    mu = F.avg_pool2d(y, 9, 1, 4)
    v = F.avg_pool2d((y - mu).square(), 9, 1, 4)
    return (y - mu) / (v + 0.001).sqrt()

def unit(raw):return resize(((raw+1)/2).clamp(0,1))
def read_mask(p):
    with Image.open(p) as im:
        if im.size!=(256,256):raise ValueError('Mask size mismatch')
        a=np.asarray(im.convert('L'))>127
    return torch.from_numpy(a.copy())[None,None]

def load_case(k):
    d=torch.load(SOURCE/'cache'/(k+'.pt'),map_location='cpu',weights_only=True)
    folder=STUDY_ROOT/'trial_A/inputs'/k
    for name in ['eval_core','eval_refmask','target','protected']:
        mask=read_mask(folder/(name+'.png'))
        if name in ['target','protected']:
            assert torch.equal(d[name].bool(),mask), (k,name)
        d[name]=mask
    for name in ['core','refmask']:d[name]=d[name].bool()
    for name in ['eval_core','eval_refmask']:assert d[name].sum()>=100
    return d

def quantile_color(im,ref,mask,refmask):
    l=lab(im)[0,:,mask[0,0].bool()];r=lab(ref)[0,:,refmask[0,0].bool()]
    q=torch.linspace(.1,.9,9,device=im.device)
    diff=(torch.quantile(l,q,dim=1)-torch.quantile(r,q,dim=1)).abs()
    return diff[:,0].mean(),diff[:,1:].mean()

def masked_mae(a,b,m):return ((a-b).abs()*m.float()).sum()/(a.shape[1]*m.sum().clamp_min(1))

def shape_metrics(im,d):
    labels=segment(align.to_bisenet(im))
    pred=F.interpolate((labels==13).float().reshape(1,1,*labels.shape[-2:]),(256,256),mode='nearest')>.5
    target=d['target']
    return dict(target_mask_IoU=float((pred&target).sum()/(pred|target).sum().clamp_min(1)),
                target_missing_fraction=float((target&~pred).sum()/target.sum().clamp_min(1)))


# Frozen differentiable evaluators: reuse exact parser weights, load original ArcFace.
from models.CtrlHair.external_code.face_parsing.my_parsing_util import FaceParsing, FaceParsing_tensor
from losses.pp_losses import Backbone
from huggingface_hub import hf_hub_download
id_path=Path('pretrained_models/ArcFace/ir_se50.pth')
if not id_path.is_file():
    downloaded=hf_hub_download(repo_id='AIRI-Institute/HairFastGAN',filename=str(id_path))
    id_path.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(downloaded,id_path)
id_net=Backbone(input_size=112,num_layers=50,drop_ratio=.6,mode='ir_se')
id_net.load_state_dict(torch.load(id_path,map_location='cpu',weights_only=True),strict=True)
id_net=id_net.cuda().eval().requires_grad_(False)
FaceParsing_tensor.parsing_img()
parser_net=FaceParsing.bise_net
if parser_net is None:raise RuntimeError('Original BiSeNet is not initialized')
parser_net.eval().requires_grad_(False)
assert FaceParsing.label_list[17]=='hair' # raw parser label=17; remapped target label=13

def id_features(im):
    x=F.interpolate(im,size=(256,256),mode='bilinear',align_corners=False)*2-1
    x=F.adaptive_avg_pool2d(x[:,:,35:223,32:220],(112,112))
    feats=id_net(x)
    return F.normalize(feats,dim=1)

def constraint_terms(im,d,fx):
    # Do NOT call get_segmentation: its no_grad + argmax would sever training gradients.
    logits=checkpoint(lambda x:parser_net(align.to_bisenet(x))[0],im,use_reentrant=False) if torch.is_grad_enabled() else parser_net(align.to_bisenet(im))[0]
    other=torch.cat((logits[:,:17],logits[:,18:]),1)
    binary=logits[:,17:18]-torch.logsumexp(other,dim=1,keepdim=True)
    target=F.interpolate(d['target'].float(),size=binary.shape[-2:],mode='nearest')
    if target.sum()==0 or (1-target).sum()==0:raise RuntimeError('Degenerate shape target')
    prob=binary.sigmoid()
    bce=F.binary_cross_entropy_with_logits(binary,target,reduction='none')
    balanced=.5*((bce*target).sum()/target.sum()+(bce*(1-target)).sum()/(1-target).sum())
    dice=1-(2*(prob*target).sum()+1)/(prob.sum()+target.sum()+1)
    embedding=checkpoint(id_features,im,use_reentrant=False) if torch.is_grad_enabled() else id_features(im)
    identity=(1-(embedding*fx['face_embedding']).sum(1)).mean()
    return identity,balanced+dice,balanced,dice

def to_size(x,size):
    return F.interpolate(x,size=(size,size),mode='bilinear',align_corners=False)

def detail_terms(full,d,fx):
    values={}
    for size in [256,512]:
        x=to_size(full,size)
        teacher=fx['pre'] if size==256 else fx['pre_512']
        mask=F.interpolate(d['eval_core'].float(),size=(size,size),mode='nearest')
        inner=-F.max_pool2d(-mask,9,1,4)
        if inner.sum()<64:raise RuntimeError('Insufficient interior hair pixels at '+str(size))
        values[size]=masked_mae(detail(x),detail(teacher),inner)
    return values


from scipy.ndimage import binary_erosion,binary_dilation
from skimage.metrics import structural_similarity
_basic_shape_metrics=shape_metrics
def shape_metrics(im,d):
    result=_basic_shape_metrics(im,d)
    labels=segment(align.to_bisenet(im));p=F.interpolate((labels==13).float().reshape(1,1,*labels.shape[-2:]),(256,256),mode='nearest')[0,0].cpu().numpy()>.5
    t=d['target'][0,0].cpu().numpy()>.5
    result['target_Dice']=float(2*(p&t).sum()/max(p.sum()+t.sum(),1))
    a=p^binary_erosion(p);bnd=t^binary_erosion(t)
    prec=(a&binary_dilation(bnd,iterations=2)).sum()/max(a.sum(),1)
    rec=(bnd&binary_dilation(a,iterations=2)).sum()/max(bnd.sum(),1)
    result['boundary_F1_2px']=float(2*prec*rec/max(prec+rec,1e-12))
    return result

def extra_metrics(im,d):
    x=im[0].permute(1,2,0).cpu().numpy();y=d['face'][0].permute(1,2,0).cpu().numpy()
    mask=d['protected'][0,0].cpu().numpy()>.5
    _,ss=structural_similarity(x,y,data_range=1.,channel_axis=2,full=True)
    safe=binary_erosion(mask,iterations=3)
    if not safe.any():raise ValueError('No protected SSIM interior')
    mse=float(((x-y)**2)[mask].mean())
    return dict(protected_PSNR=float(-10*np.log10(max(mse,1e-12))),protected_SSIM=float(ss[safe].mean()),
        any_channel_255_fraction=float((im[0,:,d['eval_core'][0,0]].max(0).values>=1-0.5/255).float().mean()))


In [ ]:
# คืนพื้นที่ GPU ของตัวสร้าง HairFastGAN; คงเฉพาะ evaluator ที่จำเป็น
for module in [enc,pp,G,align,hair_fast.embed.encoder,hair_fast.embed.e4e]:module.cpu()
id_net.cuda().eval().requires_grad_(False);parser_net.cuda().eval().requires_grad_(False)
gc.collect();torch.cuda.empty_cache()


## 3. โหลด Barbershop pretrained และฟังก์ชัน render
ใช้ latent preprocessing ที่บันทึกไว้แล้ว ไม่มี optimizer และไม่มีการฝึก

In [ ]:
BARBER_REPO=Path('/content/Barbershop_epoch30_review')
BARBER_REVISION='088d7d0f0623b348732fc15d8fbdea8b8cb730c7'
if not (BARBER_REPO/'.git').is_dir():
    subprocess.run(['git','clone','https://github.com/ZPdesu/Barbershop.git',str(BARBER_REPO)],check=True)
subprocess.run(['git','-C',str(BARBER_REPO),'checkout',BARBER_REVISION],check=True)
for kind in ['fused_act','upfirdn2d']:
    relative='models/stylegan2/op/'+kind+'.py'
    original=subprocess.check_output(['git','-C',str(BARBER_REPO),'show',BARBER_REVISION+':'+relative],text=True)
    (BARBER_REPO/relative).write_text(_native_stylegan_source(original,kind))


In [ ]:
# Load only the Barbershop synthesis module under a private package name.
# This avoids replacing HairFastGAN's 'models' or 'utils' modules.
import importlib
private=Path('/content/bs_ft_private');package=private/'bs_ft_synthesis';package.mkdir(parents=True,exist_ok=True)
(package/'__init__.py').write_text('')
shutil.copytree(BARBER_REPO/'models/stylegan2/op',package/'op',dirs_exist_ok=True)
model_source=(BARBER_REPO/'models/stylegan2/model.py').read_text().replace('from models.stylegan2.op import','from .op import')
(package/'model.py').write_text(model_source)
if str(private) not in sys.path:sys.path.insert(0,str(private))
BSGenerator=importlib.import_module('bs_ft_synthesis.model').Generator
bs=BSGenerator(1024,512,8,channel_multiplier=2)
init=torch.load(BARBER_WEIGHTS/'ffhq.pt',map_location='cpu',weights_only=False)['g_ema']
bs.load_state_dict(init,strict=True);del init
bs.eval().requires_grad_(False)
for module in list(bs.convs)[6:]+list(bs.to_rgbs)[3:]:module.requires_grad_(True)
NAMED=dict(bs.named_parameters());NAMES=[n for n,p in NAMED.items() if p.requires_grad]
CONV_NAMES=[n for n in NAMES if n.startswith('convs.')];RGB_NAMES=[n for n in NAMES if n.startswith('to_rgbs.')]
assert CONV_NAMES and RGB_NAMES and set(NAMES)==set(CONV_NAMES+RGB_NAMES)
INITIAL={n:NAMED[n].detach().cpu().clone() for n in NAMES}
def state_hash(values):
    h=hashlib.sha256()
    for n,v in sorted(values.items()):h.update(n.encode());h.update(v.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()
def frozen_hash():return state_hash({n:v for n,v in bs.state_dict().items() if n not in NAMES})
FROZEN_HASH=frozen_hash();INITIAL_HASH=state_hash(INITIAL)
bs.cuda();id_net.cuda().eval().requires_grad_(False);parser_net.cuda().eval().requires_grad_(False)
torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False

def load_trainable(values):
    assert set(values)==set(NAMES)
    with torch.no_grad():
        for n in NAMES:NAMED[n].copy_(values[n].to(NAMED[n]))
def trainable_state():return {n:NAMED[n].detach().cpu().clone() for n in NAMES}
def bs_render(S,feat,grad=False):
    # Equivalent to official generator start_layer=4/end_layer=8; checkpoint each resolution.
    out=feat;skip=None
    for j in range(3,len(bs.to_rgbs)):
        i=1+2*j
        def block(x,w,rgb,j=j,i=i):
            x=bs.convs[2*j](x,w[:,i],noise=getattr(bs.noises,f'noise_{1+2*j}'))
            x=bs.convs[2*j+1](x,w[:,i+1],noise=getattr(bs.noises,f'noise_{2+2*j}'))
            return x,bs.to_rgbs[j](x,w[:,i+2],rgb)
        out,skip=checkpoint(block,out,S,skip,use_reentrant=False) if grad else block(out,S,skip)
    return skip

def case_data(key,split):
    assert split=='validation' and key in VAL_KEYS
    d=load_case(key)
    # Keep only common supervision tensors, not HairFastGAN latent variables.
    return gpu({k:d[k] for k in ['face','ref','target','protected','eval_core','eval_refmask']})
FRONT_INDEX={}
def front_record(key,split):
    if split not in FRONT_INDEX:
        records=json.loads((BARBER_ROOT/split/'results.json').read_text())
        FRONT_INDEX[split]={r['case']:r for r in records}
    return FRONT_INDEX[split][key]
def latent(key,split):
    rec=front_record(key,split);folder=Path(rec['final_png']).parent/'Blend_realistic'
    paths=list(folder.glob('*.npz'));assert len(paths)==1
    with np.load(paths[0]) as z:S=torch.from_numpy(z['latent_in'].copy()).cuda();feat=torch.from_numpy(z['latent_F'].copy()).cuda()
    return S.detach(),feat.detach()
def reference(key,split,d):
    rec=front_record(key,split);full=torch.from_numpy(np.load(rec['final_float'])).cuda()
    with torch.no_grad():return dict(pre=to_size(full,256),pre_512=to_size(full,512),face_embedding=id_features(d['face']))

def ft_objective(raw,d,fx):
    full=((raw+1)/2).clamp(0,1);im=to_size(full,256)
    L,ab=quantile_color(im,d['ref'],d['eval_core'],d['eval_refmask'])
    identity,shape,_,_=constraint_terms(im,d,fx)
    protected=masked_mae(im,d['face'],d['protected']);details=detail_terms(full,d,fx)
    texture=torch.stack(list(details.values())).mean();color=(L+ab)/100
    w=FT_PROTOCOL['weights']
    loss=w['final_color']*color+w['final_shape']*shape+w['identity']*identity+w['protected']*protected+w['detail']*texture
    return loss,dict(loss=loss,L_quantile=L,ab_quantile=ab,shape_loss=shape,identity_cosine=1-identity,
        protected_RGB_MAE_vs_input=protected,detail_512_MAE_vs_own_baseline=details[512])


bs.requires_grad_(False)
print("Barbershop baseline and evaluators ready; no training run")


## 4. ประเมินและสร้างรายงาน
ประเมิน 45 ผลลัพธ์: baseline + A–D × validation 9 คู่ กราฟ error bar แสดง SD ระหว่างภาพ ไม่ใช่ confidence interval

ตรวจเทียบกับ log epoch30 เดิม และแสดงว่าผ่าน/ไม่ผ่านเกณฑ์เดิมข้อไหน โดยไม่เปลี่ยนเกณฑ์หรือ checkpoint ที่ล็อกไว้

In [ ]:
# Standalone review: run all preceding cells in this notebook.
# Evaluation only: no optimizer, no backward, no checkpoint selection and no test access.
from pathlib import Path
import json, hashlib, shutil, time, uuid, gc
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image
from IPython.display import display

required=['FT_ROOT','FT_ID','FT_PROTOCOL','FT_TRIALS','VAL_KEYS','BARBER_ROOT','BARBER_CASES','CROPS',
          'SOURCE','STUDY_ROOT','bs','NAMED','NAMES','INITIAL','INITIAL_HASH','FROZEN_HASH','id_net','parser_net',
          'load_trainable','trainable_state','state_hash','frozen_hash','case_data','reference','latent',
          'bs_render','ft_objective','to_size','shape_metrics','extra_metrics','saveim']
missing=[n for n in required if n not in globals()]
if missing:
    raise RuntimeError('Run all preceding setup cells in this standalone notebook. Missing: '+', '.join(missing)+'. This review does not reconstruct the training environment or run training automatically.')
assert torch.cuda.is_available(),'GPU required for checkpoint evaluation'
assert len(VAL_KEYS)==9 and len(set(VAL_KEYS))==9
assert Path(BARBER_ROOT).resolve()==(Path(FT_ROOT)/'front_end').resolve()
assert [r['key'] for r in BARBER_CASES['validation']]==list(VAL_KEYS)
assert {s['name'] for s in FT_TRIALS}==set('ABCD')
assert json.loads((Path(FT_ROOT)/'protocol.json').read_text())==FT_PROTOCOL
assert state_hash(INITIAL)==INITIAL_HASH

def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(8*1024**2),b''):h.update(block)
    return h.hexdigest()

def load_review_checkpoint(path,spec):
    ck=torch.load(path,map_location='cpu',weights_only=True)
    assert ck['format']=='barbershop_shared_synthesis_ft_v2'
    assert ck['protocol_id']==FT_ID and ck['trial']==spec
    assert ck['epoch']==30 and ck['initial_sha256']==INITIAL_HASH
    assert ck['pretrained_sha256']==FT_PROTOCOL['ffhq_sha256']
    assert set(ck['state_dict'])==set(NAMES)
    assert all(torch.isfinite(v).all() for v in ck['state_dict'].values())
    if path.name=='last.pt':assert len(ck['steps'])==4500
    assert state_hash(ck['state_dict'])!=INITIAL_HASH,'Epoch30 weights equal initialization; inspect training before proceeding'
    return ck

# Verify the historical validation cache and masks before evaluation.
for key in VAL_KEYS:
    assert file_sha(Path(SOURCE)/'cache'/(key+'.pt'))==FT_PROTOCOL['source_cache_hashes'][key], 'Validation cache changed: '+key
    for name,digest in FT_PROTOCOL['mask_hashes'][key].items():
        assert file_sha(Path(STUDY_ROOT)/'trial_A/inputs'/key/(name+'.png'))==digest, 'Validation mask changed: '+key+'/'+name

# Verify ALL candidate files before evaluating any images.
checkpoint_paths={};checkpoint_info={}
for spec in FT_TRIALS:
    t=spec['name'];folder=Path(FT_ROOT)/('trial_'+t)
    manifest=json.loads((folder/'manifest.json').read_text())
    assert manifest['completed_epoch']==30 and manifest['weight_updates']==4500
    path=folder/'final_epoch30.pt'
    if not path.is_file():
        path=folder/'last.pt'
        print(t,': final_epoch30.pt missing; validating last.pt is genuinely epoch30.')
    if not path.is_file():raise FileNotFoundError(str(path)+'; do not substitute best.pt (which may be epoch0).')
    ck=load_review_checkpoint(path,spec)
    checkpoint_paths[t]=path
    checkpoint_info[t]=dict(path=str(path),sha256=file_sha(path),epoch=30,weight_sha256=state_hash(ck['state_dict']),original_best_epoch=manifest['best_epoch'])
    del ck
protected=[Path(FT_ROOT)/'protocol.json',Path(FT_ROOT)/'locked_before_test.json']
for t in 'ABCD':protected += [p for p in (Path(FT_ROOT)/('trial_'+t)).glob('*.pt')]
protected={str(p):file_sha(p) for p in protected if p.is_file()}

stamp=time.strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6]
REVIEW_OUT=Path(FT_ROOT)/('validation_epoch30_review_'+stamp)
REVIEW_OUT.mkdir(exist_ok=False)
(REVIEW_OUT/'review_protocol.json').write_text(json.dumps(dict(
    study=FT_ID,review_type='Exploratory validation-only baseline versus fixed epoch30',
    validation_keys=list(VAL_KEYS),models=['Baseline','A','B','C','D'],
    checkpoint_info=checkpoint_info,selection_changed=False,test_evaluated=False,
    note='Epoch30 was fixed in advance of this review; no new best checkpoint is selected. Detail measures change from own baseline, not independent quality. IoU uses HFG-derived targets.'),indent=2))

# Preserve the user's in-memory model state and the original checkpoints.
entry_weights=trainable_state();entry_hash=state_hash(entry_weights)
entry_device=next(bs.parameters()).device
entry_flags=[(m,m.training) for m in bs.modules()]
entry_grad={n:p.requires_grad for n,p in bs.named_parameters()}
evaluator_states=[(m,next(m.parameters()).device,[(sub,sub.training) for sub in m.modules()]) for m in [id_net,parser_net]]
(REVIEW_OUT/'runtime_environment.json').write_text(json.dumps(dict(python=sys.version,torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),hairfast_commit=REPO_COMMIT,barbershop_commit=BARBER_REVISION),indent=2))
rows=[]
try:
    bs.cuda().eval().requires_grad_(False);id_net.cuda().eval();parser_net.cuda().eval()
    assert frozen_hash()==FROZEN_HASH
    with torch.no_grad():
        for model in ['Baseline','A','B','C','D']:
            if model=='Baseline':load_trainable(INITIAL);epoch=0
            else:
                spec=next(s for s in FT_TRIALS if s['name']==model)
                assert file_sha(checkpoint_paths[model])==checkpoint_info[model]['sha256']
                ck=load_review_checkpoint(checkpoint_paths[model],spec)
                load_trainable(ck['state_dict']);epoch=30;del ck
            before=state_hash(trainable_state())
            for key in VAL_KEYS:
                d=case_data(key,'validation');fx=reference(key,'validation',d)
                S,feat=latent(key,'validation');raw=bs_render(S,feat,grad=False)
                loss,vals=ft_objective(raw,d,fx)
                im=to_size(((raw+1)/2).clamp(0,1),256)
                values={k:float(v) for k,v in vals.items()}
                values.update(shape_metrics(im,d));values.update(extra_metrics(im,d))
                assert all(np.isfinite(v) for v in values.values())
                rows.append(dict(model=model,epoch=epoch,case=key,split='validation',**values))
                folder=REVIEW_OUT/'images'/key;folder.mkdir(parents=True,exist_ok=True)
                saveim((raw+1)/2,folder/(model+'.png'))
                if model=='Baseline':
                    pair=next(r for r in BARBER_CASES['validation'] if r['key']==key)
                    for role,name in [('face','face'),('shape','shape'),('color','ref')]:
                        shutil.copy2(Path(CROPS)/(pair[role]+'.png'),folder/(name+'.png'))
                    saveim(d['target'].float().repeat(1,3,1,1),folder/'target.png')
                    saveim(d['protected'].float().repeat(1,3,1,1),folder/'protected.png')
                pd.DataFrame(rows).to_csv(REVIEW_OUT/'per_case_metrics.partial.csv',index=False)
                print(f'Validation review: {model}, epoch {epoch}, {key}',flush=True)
                del d,fx,S,feat,raw,im,loss,vals
            assert state_hash(trainable_state())==before,'Evaluation changed shared weights'
            assert frozen_hash()==FROZEN_HASH
finally:
    load_trainable(entry_weights)
    for n,p in bs.named_parameters():p.requires_grad_(entry_grad[n])
    for module,flag in entry_flags:module.training=flag
    bs.to(entry_device)
    for module,device,flags in evaluator_states:
        module.to(device)
        for sub,flag in flags:sub.training=flag
    assert state_hash(trainable_state())==entry_hash,'In-memory weights not restored'
    for path,digest in protected.items():assert file_sha(path)==digest,'Original file changed: '+path
    del entry_weights;gc.collect();torch.cuda.empty_cache()

frame=pd.DataFrame(rows)
assert len(frame)==45 and not frame.duplicated(['model','case']).any()
assert set(frame[frame.model=='Baseline'].epoch)=={0}
assert set(frame[frame.model!='Baseline'].epoch)=={30}
frame.to_csv(REVIEW_OUT/'per_case_metrics.csv',index=False)
(REVIEW_OUT/'per_case_metrics.partial.csv').unlink()
order=['Baseline','A','B','C','D'];colors=['#7B8494','#3178C6','#098C79','#DB9031','#9966B3']
metrics=[('L_quantile','Lightness error L ↓'),('ab_quantile','Color error ab ↓'),('target_mask_IoU','Hair-mask IoU ↑'),
         ('identity_cosine','Face identity cosine ↑'),('protected_RGB_MAE_vs_input','Protected-region MAE ↓'),
         ('detail_512_MAE_vs_own_baseline','Detail MAE vs own baseline ↓'),('boundary_F1_2px','Hair boundary F1 ↑'),
         ('protected_SSIM','Protected-region SSIM ↑'),('protected_PSNR','Protected-region PSNR ↑')]
cols=[k for k,_ in metrics]
mean=frame.groupby('model')[cols].mean().reindex(order);sd=frame.groupby('model')[cols].std(ddof=1).reindex(order)
mean.to_csv(REVIEW_OUT/'mean_metrics.csv');sd.to_csv(REVIEW_OUT/'sd_across_cases.csv')
# Paired deltas: negative is better for error metrics; positive is better for similarities.
base=frame[frame.model=='Baseline'].set_index('case')
paired=[]
for t in 'ABCD':
    current=frame[frame.model==t].set_index('case').reindex(base.index)
    for key in base.index:paired.append(dict(model=t,case=key,**{k:float(current.loc[key,k]-base.loc[key,k]) for k in cols}))
pd.DataFrame(paired).to_csv(REVIEW_OUT/'paired_deltas_vs_baseline.csv',index=False)
(mean-mean.loc['Baseline']).to_csv(REVIEW_OUT/'mean_delta_vs_baseline.csv')

# Diagnose the ORIGINAL gates. Do not change their thresholds or select new best checkpoints.
g=FT_PROTOCOL['gates'];baseline=mean.loc['Baseline'];gate_rows=[]
for t in 'ABCD':
    v=mean.loc[t]
    checks=dict(color=v.L_quantile+v.ab_quantile<=baseline.L_quantile+baseline.ab_quantile,
        shape=v.target_mask_IoU>=baseline.target_mask_IoU,
        identity=v.identity_cosine>=baseline.identity_cosine-g['identity_drop'],
        protected=v.protected_RGB_MAE_vs_input<=baseline.protected_RGB_MAE_vs_input+g['protected_increase'],
        detail=v.detail_512_MAE_vs_own_baseline<=baseline.detail_512_MAE_vs_own_baseline+g['detail_absolute_increase'])
    gate_rows.append(dict(model=t,epoch=30,**{k:bool(v) for k,v in checks.items()},all_gates_pass=all(checks.values())))
pd.DataFrame(gate_rows).to_csv(REVIEW_OUT/'original_gate_diagnostics.csv',index=False)
# Compare new epoch30 evaluation with historical validation logs, without demanding bitwise repeatability.
diffs=[]
for t in 'ABCD':
    histpath=Path(FT_ROOT)/('trial_'+t)/'validation_per_case.csv'
    if histpath.is_file():
        historical=pd.read_csv(histpath);historical=historical[historical.epoch==30]
        joined=frame[frame.model==t].merge(historical,on='case',suffixes=('_now','_history'),validate='one_to_one')
        assert len(joined)==9
        for k in cols:diffs.append(dict(model=t,metric=k,max_abs_delta=float((joined[k+'_now']-joined[k+'_history']).abs().max())))
pd.DataFrame(diffs).to_csv(REVIEW_OUT/'reevaluation_vs_recorded_epoch30.csv',index=False)

plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'figure.facecolor':'white'})
with PdfPages(REVIEW_OUT/'validation_epoch30_review.pdf') as pdf:
    fig,axs=plt.subplots(3,3,figsize=(15,12))
    for ax,(k,title) in zip(axs.flat,metrics):
        ax.errorbar(range(5),mean[k],yerr=sd[k],fmt='none',color='#667085',capsize=4)
        ax.scatter(range(5),mean[k],c=colors,s=70,zorder=4)
        ax.set_xticks(range(5),['Baseline\nep 0']+[f'{t}\nep 30' for t in 'ABCD'])
        ax.set_title(title,loc='left',fontsize=11);ax.grid(alpha=.2);ax.spines[['top','right']].set_visible(False)
    fig.suptitle('Validation review | Baseline vs fixed epoch 30 | Mean ± SD, n=9',fontsize=17)
    fig.text(.04,.01,'Exploratory review, not a new checkpoint selection. Detail difference does not establish quality loss. IoU uses HFG-derived targets.',fontsize=9)
    fig.tight_layout(rect=(0,.035,1,.96));fig.savefig(REVIEW_OUT/'metrics_epoch30.png',dpi=180);pdf.savefig(fig);plt.show();plt.close(fig)
    for key in VAL_KEYS:
        folder=REVIEW_OUT/'images'/key
        fig,axs=plt.subplots(2,4,figsize=(16,9))
        panels=[('face','Input face'),('shape','Shape reference'),('ref','Color reference'),('Baseline','Pretrained baseline | epoch 0')]+[(t,f'Trial {t} | epoch 30') for t in 'ABCD']
        for ax,(name,label) in zip(axs.flat,panels):
            with Image.open(folder/(name+'.png')) as image:ax.imshow(image)
            ax.set_title(label,fontsize=11);ax.axis('off')
        fig.suptitle(key+' | Validation only | Fixed epoch30 review',fontsize=17);fig.tight_layout()
        fig.savefig(REVIEW_OUT/(key+'_comparison.png'),dpi=160);pdf.savefig(fig);plt.show();plt.close(fig)
        # Full-resolution images above remain available; these equal crops aid inspection of local changes.
        fig,axs=plt.subplots(2,5,figsize=(16,7))
        for col,model in enumerate(order):
            with Image.open(folder/(model+'.png')) as image:
                w,h=image.size
                for row,(name,box) in enumerate([('Hair / forehead',(0,0,w,int(h*.60))),('Face / ears',(int(w*.18),int(h*.25),int(w*.82),int(h*.85)))]):
                    axs[row,col].imshow(image.crop(box));axs[row,col].set_title(model+' | '+name,fontsize=10);axs[row,col].axis('off')
        fig.suptitle(key+' | Identical crop windows; no enhancement applied');fig.tight_layout()
        fig.savefig(REVIEW_OUT/(key+'_detail_crops.png'),dpi=160);pdf.savefig(fig);plt.close(fig)

# Blank manual review sheet: do not auto-label changed texture as degraded detail.
review_rows=[dict(case=k,model=t,forehead_edge='',hair_texture='',face_and_ears='',color_spill='',notes='') for k in VAL_KEYS for t in 'ABCD']
pd.DataFrame(review_rows).to_csv(REVIEW_OUT/'visual_review_blank.csv',index=False)
(REVIEW_OUT/'README_TH.txt').write_text('''ตรวจ validation ของ baseline epoch0 เทียบ A-D epoch30 ที่กำหนดไว้ล่วงหน้า
ไม่ฝึกใหม่ ไม่ประเมิน test ไม่แก้ best.pt ไม่เลือก best checkpoint ใหม่
ดู metrics_epoch30.png, ภาพ comparison และ detail_crops คู่กัน แล้วบันทึก visual_review_blank.csv
ค่า detail เป็นความต่างจาก baseline ไม่ใช่ความถูกต้องของรายละเอียด
original_gate_diagnostics.csv อธิบายเกณฑ์เดิม ไม่ใช่การอนุมัติเกณฑ์ใหม่
reevaluation_vs_recorded_epoch30.csv ตรวจความต่างระหว่างการประเมินรอบนี้และ log ตอนฝึก
รายงานนี้เป็นการตรวจเพิ่มเติมหลังเห็นผลเดิม ไม่ใช่ผลยืนยันบน test ใหม่
''',encoding='utf-8')
(REVIEW_OUT/'completion.json').write_text(json.dumps(dict(status='complete',split='validation',cases=9,metric_rows=45,
    epochs={'Baseline':0,**{t:30 for t in 'ABCD'}},original_files_unchanged=True,in_memory_weights_restored=True,
    no_training=True,no_test_evaluation=True,no_reselection=True),indent=2))
display(mean);display(pd.DataFrame(gate_rows))
REVIEW_ZIP_EPOCH30=shutil.make_archive(str(REVIEW_OUT),'zip',root_dir=REVIEW_OUT)
print('Saved review:',REVIEW_ZIP_EPOCH30)
from google.colab import files
files.download(REVIEW_ZIP_EPOCH30)
